In [46]:
import pandas as pd
# Remember downloading spacy according to how they say on their website.
import spacy
from collections import Counter

In [47]:
# Load our dataframe
df = pd.read_csv("lyrics_chunks_enriched.csv")

In [48]:
# 把同一个艺人的所有歌词合并成一个长文本
artist_lyrics = df.groupby('artist')['text'].apply(lambda x: ' '.join(x)).reset_index()

# Rename columns
artist_lyrics.columns = ['artist', 'combined_text']

print(artist_lyrics.head())
print(f"\n共有 {len(artist_lyrics)} 位艺人")

       artist                                      combined_text
0         141  当我站在这条路我就知道弱肉强食\n拼命变得强大 我要以一敌十\nshout out to谦子...
1  404 Rapper  Shooter：\n我的天 帅哥在我身边\n要把他放心尖 演电影的画面\n他点燃我欲火 我不...
2    APMOZART  出品 : LeaderNotFollow\nI’ m forever\nI’ m forev...
3          AR  别墅里面唱k\n水池里面银龙鱼\n我送阿叔茶具\n他研墨下笔直接给我四个字\n大展鸿图大师亲...
4      Ansr J  Prod. by HARIKIRI\n⻢思唯:\n珠宝首饰到处丢 我从游泳池头醒来\n出门之...

共有 241 位艺人


In [49]:
# import spacy
# from spacy.cli import download
#
# try:
#     # Attempt to download the specified SpaCy model
#     download("zh_core_web_lg")
#     print("Model 'zh_core_web_lg' successfully downloaded.")
# except Exception as e:
#     print(f"Failed to download the SpaCy model. Error: {e}")
#     print("Please ensure you have an active internet connection and sufficient permissions.")

In [50]:
# Load Chinese (zh) spacy
nlp = spacy.load('zh_core_web_lg')

# Quick test of the package
doc = nlp("周杰伦在北京唱歌")
for ent in doc.ents:
    print(ent.text, ent.label_)

周杰伦 PERSON
北京 GPE


In [51]:
print(f"数据形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")
print("\n前5行数据:")
print(df.head())

数据形状: (26833, 5)
列名: ['artist', 'song_id', 'song_title', 'chunk_id', 'text']

前5行数据:
  artist   song_id  song_title  chunk_id  \
0    周士爵  周士爵_0000  让我们平静地走进黑夜         0   
1    周士爵  周士爵_0001         女人啊         0   
2    周士爵  周士爵_0002        悲极生乐         0   
3    周士爵  周士爵_0003        找一颗心         0   
4    周士爵  周士爵_0004     去爱去哭去疑惑         0   

                                                text  
0  岁月之初怀有很多梦想\n那清晰又 美妙的构想\n向世界放飞我的幻想\n情谊在心中 朋友在远方...  
1  人啊，你深渊的底 到底在哪里\n谁，也测不出来 所以才无奈\n可这旋律刚一响起，眼睛就落下了...  
2  童声 : 石梓墨/孙一涵/吕翼娆\n巨大的怪物\n它也会哭\n一滴泪就是一场雨\n表情无辜\...  
3  冰激凌\n泻立停\n黑猫喝\n白猫牌洗洁精\n话太冷\n冻成冰\n瑟瑟发抖\n用纸杯里的玻璃...  
4  小老虎：别信这一套\n她犹豫她深呼吸\n你只是不够好\n为了拔这根刺\n花钱治疗心里的病\n...  


In [52]:
print(f"共有 {len(artist_lyrics)} 位艺人")
print("\n艺人列表（前10位）:")
print(artist_lyrics['artist'].head(10).tolist())

# Count the total number of characters(not chinese) in the first song of the indexed artist
print(f"\n示例：{artist_lyrics.iloc[2]['artist']} 的歌词总字符数: {len(artist_lyrics.iloc[2]['combined_text'])}")

print("\n艺人列表（后10位）:")
print(artist_lyrics['artist'].tail(10).tolist())

共有 241 位艺人

艺人列表（前10位）:
['141', '404 Rapper', 'APMOZART', 'AR', 'Ansr J', 'Athree', 'Ayo97', 'BeCareful', 'Blow Fever', 'BossWang']

示例：APMOZART 的歌词总字符数: 73722

艺人列表（后10位）:
['黄景行', '黑眼圈', '黑螺丝', '黑麦', '默示船政', '龙井说唱 孙旭', '龙井说唱 孙骁', '龙井说唱益晗', '龙泽宇', '龙胆紫']


In [53]:
# This where we use collections library

# Test Artist variable creation
test_artist = artist_lyrics.iloc[3]
print(f"测试艺人: {test_artist['artist']}")
print(f"文本长度: {len(test_artist['combined_text'])} 字符")

测试艺人: AR
文本长度: 88864 字符


In [54]:
# NER

# Set input for entity recognition
doc = nlp(test_artist['combined_text'])

# Set up for-loop to Extract entities
entities = []
for ent in doc.ents:
    entities.append({
        'text': ent.text,
        'label': ent.label_
    })

print(f"提取到 {len(entities)} 个实体")

提取到 1513 个实体


IndentationError: expected an indented block (2435969062.py, line 6)

In [70]:

# Now run NER on all artists

all_entities = []
processed_count = 0
total_artists = len(artist_lyrics)

# Process each artist's text with NER
for _, row in artist_lyrics.iterrows():
    artist = row['artist']
    text = row['combined_text']

    # Process with SpaCy NER
    doc = nlp(text)

    # Extract entities
    for ent in doc.ents:
        all_entities.append({
            'text': ent.text,
            'label': ent.label_
        })

    processed_count += 1
    if processed_count % 5 == 0 or processed_count == total_artists:
        print(f"Processed {processed_count}/{total_artists} artists ({(processed_count/total_artists)*100:.1f}%)")

print(f"\nCompleted NER processing. Extracted {len(all_entities)} entities from all artists")


Processed 5/241 artists (2.1%)
Processed 10/241 artists (4.1%)
Processed 15/241 artists (6.2%)
Processed 20/241 artists (8.3%)
Processed 25/241 artists (10.4%)
Processed 30/241 artists (12.4%)
Processed 35/241 artists (14.5%)
Processed 40/241 artists (16.6%)
Processed 45/241 artists (18.7%)
Processed 50/241 artists (20.7%)
Processed 55/241 artists (22.8%)
Processed 60/241 artists (24.9%)
Processed 65/241 artists (27.0%)
Processed 70/241 artists (29.0%)
Processed 75/241 artists (31.1%)
Processed 80/241 artists (33.2%)
Processed 85/241 artists (35.3%)
Processed 90/241 artists (37.3%)
Processed 95/241 artists (39.4%)
Processed 100/241 artists (41.5%)
Processed 105/241 artists (43.6%)
Processed 110/241 artists (45.6%)
Processed 115/241 artists (47.7%)
Processed 120/241 artists (49.8%)
Processed 125/241 artists (51.9%)
Processed 130/241 artists (53.9%)
Processed 135/241 artists (56.0%)
Processed 140/241 artists (58.1%)
Processed 145/241 artists (60.2%)
Processed 150/241 artists (62.2%)
Proc

In [72]:
# Frequency (note th
entity_counter = Counter([(e['text'], e['label']) for e in all_entities])

# The most common entities
for (text, label), count in entity_counter.most_common(50):
    print(f"{text} ({label}): {count}次")

两 (CARDINAL): 1777次
今天 (DATE): 1292次
一 (CARDINAL): 1200次
第一 (ORDINAL): 1171次
三 (CARDINAL): 992次
北京 (GPE): 991次
2 (CARDINAL): 854次
明天 (DATE): 820次
中国 (GPE): 806次
成都 (GPE): 674次
今晚 (TIME): 627次
一天 (DATE): 618次
4 (CARDINAL): 558次
晚上 (TIME): 495次
一半 (CARDINAL): 483次
中文 (LANGUAGE): 470次
wit (PERSON): 451次
夏天 (DATE): 426次
五 (CARDINAL): 413次
冬天 (DATE): 384次
昨天 (DATE): 381次
四 (CARDINAL): 355次
广东 (GPE): 354次
八 (CARDINAL): 343次
上海 (GPE): 342次
3 (CARDINAL): 339次
夜里 (TIME): 314次
重庆 (GPE): 314次
老子 (PERSON): 308次
今年 (DATE): 307次
深夜 (TIME): 304次
白天 (TIME): 303次
7 (CARDINAL): 303次
二 (CARDINAL): 293次
地球 (LOC): 293次
一年 (DATE): 291次
网易 (ORG): 260次
Everybody (ORG): 257次
Money (ORG): 256次
长沙 (GPE): 251次
OP (PERSON): 239次
夜晚 (TIME): 237次
午夜 (TIME): 230次
Prod (GPE): 228次
一秒 (TIME): 226次
王 (PERSON): 220次
十 (CARDINAL): 217次
老子们 (WORK_OF_ART): 196次
nothing (PERSON): 194次
1 (CARDINAL): 192次


# Adding in jieba here so we can have segmentation of all the words, frequency, and compare that to NER output

In [56]:
import re, jieba, numpy as np, pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer

df = pd.read_csv("lyrics_chunks_enriched.csv")

In [57]:
# Load stopwords from the file
with open("stopwords-zh.txt", "r", encoding="utf-8") as f:
    stopwords = set(line.strip() for line in f if line.strip())

In [58]:
def zh_tokenize(s: str):
    s = re.sub(r"\s+", "", str(s))  # Remove whitespace
    toks = [w for w in jieba.lcut(s, HMM=True) if w]  # Segment with Jieba
    out = []
    for w in toks:
        # Filter: stopwords, pure numbers/Latin, punctuation
        if w in stopwords:
            continue
        if re.fullmatch(r"[0-9A-Za-z]+", w):
            continue
        if re.fullmatch(r"\W+", w):
            continue
        out.append(w)
    return out

In [59]:
# Jieba frequency on 'Test Artist'
segmented_words = zh_tokenize(test_artist['combined_text'])
word_frequencies = Counter(segmented_words)

# Display the most common words with their frequencies
most_common_words = word_frequencies.most_common(50)
for word, freq in most_common_words:
    print(f"{word}: {freq}次")

# Save results to a DataFrame
freq_df = pd.DataFrame(most_common_words, columns=['Word', 'Frequency'])
print("\nWord Frequency DataFrame:\n", freq_df)

广东: 115次
哥们: 108次
丢: 88次
想: 85次
没有: 80次
说唱: 79次
没: 77次
听: 72次
里: 64次
嘅: 57次
唔: 52次
对号入座: 51次
叉烧: 49次
需要: 47次
兄弟: 42次
真的: 42次
变成: 40次
不能: 39次
刷: 39次
唱: 38次
不用: 38次
押韵: 38次
北上: 37次
爱: 36次
太: 35次
买: 32次
现在: 32次
天才: 32次
看到: 31次
像是: 30次
知道: 30次
话: 29次
有点: 28次
宜家: 28次
歌: 27次
风: 27次
继续: 27次
不会: 26次
写: 26次
左: 26次
变: 25次
北京: 25次
不要: 25次
走: 25次
东西: 25次
校长: 25次
看着: 24次
民族: 24次
打折: 24次
玩: 23次

Word Frequency DataFrame:
     Word  Frequency
0     广东        115
1     哥们        108
2      丢         88
3      想         85
4     没有         80
5     说唱         79
6      没         77
7      听         72
8      里         64
9      嘅         57
10     唔         52
11  对号入座         51
12    叉烧         49
13    需要         47
14    兄弟         42
15    真的         42
16    变成         40
17    不能         39
18     刷         39
19     唱         38
20    不用         38
21    押韵         38
22    北上         37
23     爱         36
24     太         35
25     买         32
26    现在         32
27    天才         32
28    看到

In [60]:
# Jieba frequency on all artists
# Initialize the all_words list before using it
all_words = []

# Loop through all artists and tokenize their text
for _, row in artist_lyrics.iterrows():
    all_words.extend(zh_tokenize(row['combined_text']))

# Calculate word frequencies
all_word_frequencies = Counter(all_words)

# Create a DataFrame for word frequencies
all_freq_df = pd.DataFrame(all_word_frequencies.most_common(), columns=['Word', 'Frequency'])

# Sort by frequency in descending order
jieba_df = all_freq_df.sort_values(by='Frequency', ascending=False)

# Display the top 50 words
print("Top 50 Words:")
print(jieba_df.head(50))

# Save the results to a CSV file
jieba_df.to_csv('all_artists_word_frequencies.csv', index=False, encoding='utf-8')

Top 50 Words:
   Word  Frequency
0     想      12011
1     没       9673
2    没有       9336
3     爱       7254
4     里       6461
5     走       6369
6    不会       5998
7    现在       5346
8    兄弟       5213
9    想要       4839
10   知道       4638
11    听       4591
12    太       4140
13   世界       3909
14   说唱       3824
15   真的       3789
16   老子       3654
17   不想       3603
18   时间       3560
19   音乐       3509
20   需要       3490
21   一起       3412
22   生活       3259
23   不要       3207
24   不能       3069
25   一直       2975
26   感觉       2958
27    带       2957
28   可能       2884
30   已经       2881
29   喜欢       2881
31    钱       2778
32   像是       2717
33    吃       2525
34   看到       2433
35   不用       2404
36   永远       2378
37    找       2376
38   能够       2278
39   变成       2269
40   继续       2248
41    站       2240
42    新       2225
43    耍       2203
44    懂       2202
45    ى       2179
46   最后       2160
47    快       2130
48   每天       2099
49   觉得       2090


In [61]:
jieba_df.shape

(103091, 2)

In [62]:
# NER for all artists
all_entities = []


In [73]:
# Count entity frequencies
entity_counter = Counter([(e['text'], e['label']) for e in all_entities])

# Create NER dataframe
ner_df = pd.DataFrame([
    {"word": word, "entity_type": label, "frequency": count}
    for (word, label), count in entity_counter.most_common()
])

print(f"Created NER frequency dataframe with {len(ner_df)} entities")
print(ner_df.head())

# Display top entities by frequency
print("\nTop 20 named entities by frequency:")
for i, row in ner_df.head(20).iterrows():
    print(f"{row['word']} ({row['entity_type']}): {row['frequency']} occurrences")


Created NER frequency dataframe with 29544 entities
  word entity_type  frequency
0    两    CARDINAL       1777
1   今天        DATE       1292
2    一    CARDINAL       1200
3   第一     ORDINAL       1171
4    三    CARDINAL        992

Top 20 named entities by frequency:
两 (CARDINAL): 1777 occurrences
今天 (DATE): 1292 occurrences
一 (CARDINAL): 1200 occurrences
第一 (ORDINAL): 1171 occurrences
三 (CARDINAL): 992 occurrences
北京 (GPE): 991 occurrences
2 (CARDINAL): 854 occurrences
明天 (DATE): 820 occurrences
中国 (GPE): 806 occurrences
成都 (GPE): 674 occurrences
今晚 (TIME): 627 occurrences
一天 (DATE): 618 occurrences
4 (CARDINAL): 558 occurrences
晚上 (TIME): 495 occurrences
一半 (CARDINAL): 483 occurrences
中文 (LANGUAGE): 470 occurrences
wit (PERSON): 451 occurrences
夏天 (DATE): 426 occurrences
五 (CARDINAL): 413 occurrences
冬天 (DATE): 384 occurrences


In [75]:
# Now combine the dataframes
# First, make a copy of the NER dataframe to use as our base
combined_df = ner_df.copy()

# Add a column to indicate these rows are from NER
combined_df['source'] = 'NER'

# Check which Jieba words are not in the NER dataframe
jieba_words = set(jieba_df['Word'])
ner_words = set(ner_df['word'])

# Find words that are only in Jieba
jieba_only_words = jieba_words - ner_words


Found 96026 words that are only in Jieba results


In [77]:
# Create a dataframe for Jieba-only words
jieba_only_df = jieba_df[jieba_df['Word'].isin(jieba_only_words)].copy()
jieba_only_df['entity_type'] = None  # No entity type for Jieba-only words
jieba_only_df['source'] = 'Jieba'

# Find Jieba words that are also in NER
jieba_and_ner = jieba_words.intersection(ner_words)

# Update the source column for words that appear in both
combined_df.loc[combined_df['word'].isin(jieba_and_ner), 'source'] = 'NER+Jieba'



In [80]:
jieba_only_df.rename(columns={'Word': 'word', 'Frequency': 'frequency'}, inplace=True)

In [82]:
# Combine the dataframes
final_df = pd.concat([combined_df, jieba_only_df], ignore_index=True)

print(f"Final combined dataframe has {len(final_df)} rows")
print(final_df.head())

# Optional: Save the combined dataframe
final_df.to_csv("combined_ner_jieba_frequencies.csv", index=False, encoding="utf-8")

Final combined dataframe has 125570 rows
  word entity_type  frequency     source
0    两    CARDINAL       1777  NER+Jieba
1   今天        DATE       1292  NER+Jieba
2    一    CARDINAL       1200        NER
3   第一     ORDINAL       1171  NER+Jieba
4    三    CARDINAL        992        NER


Add arbitrary classes to the entity recognition system, and update the model with new examples ( I think this is called a lexical dictionary)

In [ ]:
# Current NER entity types: CARDINAL, DATE, EVENT, FAC, GPE, LANGUAGE, LAW, LOC, MONEY, NORP, ORDINAL, ORG, PERCENT, PERSON, PRODUCT, QUANTITY, TIME, WORK_OF_ART
# There explanations:
PERSON:      People, including fictional.
NORP:        Nationalities or religious or political groups.
FAC:         Buildings, airports, highways, bridges, etc.
ORG:         Companies, agencies, institutions, etc. Brands
GPE:         Countries, cities, states.
LOC:         Non-GPE locations, mountain ranges, bodies of water.
HOM:         Non-LOC locations, e.g., streets, neighborhoods, etc
PRODUCT:     Objects, vehicles, foods, etc. (Not services.)
EVENT:       Named hurricanes, battles, wars, sports events, etc.
WORK_OF_ART: Titles of books, songs, etc.
TIME:        Times smaller than a day.
PERCENT:     Percentage, including ”%“.
MONEY:       Monetary values, including unit.
QUANTITY:    Measurements, as of weight or distance.
CARDINAL:    Numerals that do not fall under another type.

4. Cultural and Material Items

    Fashion Items: Sneakers, hoodies, caps, and jewelry ("bling").
    Automobiles: Specific models of cars associated with wealth or lifestyle.
    Substances: Alcohol and drugs (e.g., Xanax, codeine, marijuana).
    Weapons: References to firearms, often tied to themes of violence, self-defense, or crime.

5. Intangible and Technical Entities

    Slang/Vocabulary: Unique jargon such as "bars," "flow," "diss," or "hook".
    Musical Elements/Tools: Mentions of equipment or techniques like 808s, MPCs, samplers, or specific dance moves.
    Cultural References: Allusions to movies, television shows (e.g., Scarface, The Godfather), or other pop culture, particularly in relation to the mafia.


# We need to go through the NER and Jieba combined dataframe and decide what is important that should be labeled. 

# Arbitrary Classes
SpaCy provides an exceptionally efficient statistical system for NER in python, which can assign labels to groups of tokens which are contiguous. It provides a default model which can recognize a wide range of named or numerical entities, which include person, organization, language, event etc. Apart from these default entities, spaCy also gives us the liberty to add arbitrary classes to the NER model, by training the model to update it with newer trained examples.

    Load the model 1.1. spacy.load('en') --> Disable existing pipe line (nlp.disable_pipes) 1.2. spacy.blank('en') --> Added Entity Recognizer to Pipeline
    Shuffle and loop over the examples --> update the model (nlp.update)
    Save the trained model (nlp.to_disk)
    Test

# We also have to figure out for example 北京 上海等其他地名, are these from the beijing shanghai etc rappers or are artists from other cities shouting this stuff out?


1. Loads the pre-trained Chinese spaCy model
2. Creates an EntityRuler component
3. Reads entities from your CSV file
4. Adds the patterns to the ruler
5. Adds the ruler to the spaCy pipeline
6. Includes a test example to show how it works

This approach is much more straightforward when you already know the entities you want to recognize, as it doesn't require the complex training process with annotated sentences.
The key advantages are:
- No need to provide full sentences with position annotations
- Works with just a list of entities and their labels
- Can be easily updated by adding more entities to your CSV
- Combines with spaCy's existing statistical model for a hybrid approach
- Domain-specific categorization: Create labels specific to your domain (music, finance, medicine, etc.)
- Granular entity typing: Go beyond generic "PERSON" to have "ARTIST", "PRODUCER", "SONGWRITER"



In [ ]:
import spacy
from spacy.pipeline import EntityRuler
import pandas as pd

# Load the Chinese spaCy model
nlp = spacy.load("zh_core_web_lg")

# Load entity patterns from a CSV file
# Assuming the CSV has two columns: 'pattern' (contains text to identify) and 'label' (the entity type)
entity_patterns_path = "entity_patterns.csv"

patterns_df = pd.read_csv(entity_patterns_path)

# Convert patterns DataFrame to a list of dictionaries
patterns = [{"label": row['label'], "pattern": row['pattern']} for _, row in patterns_df.iterrows()]

# Create the EntityRuler and add the patterns
# The 'before' here is necessary to ensuring our patterns take precedence over spacy's stats model
ruler = nlp.add_pipe("entity_ruler", before="ner") #, config={"overwrite_ents": True}) # This ensures our patterns overwrite overlapping entities

ruler.add_patterns(patterns)

# Test the EntityRuler
test_text = "周杰伦在北京开了一场非常重要的演唱会。"
doc = nlp(test_text)

# Print detected entities
print("Detected Entities:")
for ent in doc.ents:
    print(f"{ent.text} ({ent.label_})")